In [ ]:
import requests
from tqdm import tqdm
import os
from llama_cpp import Llama
import time
import psutil

# Your selected model
model_url = "https://huggingface.co/gaianet/Mistral-7B-Instruct-v0.3-GGUF/resolve/main/Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"
filename = "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"



# Download with progress bar
response = requests.get(model_url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(filename, 'wb') as file, tqdm(
        desc=filename,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
        colour='green'
    ) as bar:
    for data in response.iter_content(chunk_size=1024):
        bar.update(len(data))
        file.write(data)

print("Done")
print(f"File saved as: {filename}")
print(f"File size: {os.path.getsize(filename) / (1024**3):.2f} GB")

In [3]:
full_prompt = """<|system|>
You are a senior social policy analyst for a UK local authority. Analyze this statistical summary of a regional population survey and provide strategic insights for policymakers.

<|user|>
Here's a statistical summary from our 2024 Regional Demographic & Wellbeing Survey (sample size: 10,000 respondents) for a mixed urban-rural region in the UK. Please analyze and provide insights: DEMOGRAPHIC STATISTICS SUMMARY - 2024 (UK REGION)

POPULATION OVERVIEW:
┌──────────────────────────────┬─────────────┬─────────────────┐
│ Metric │ Value │ UK Average │
├──────────────────────────────┼─────────────┼─────────────────┤
│ Total Population │ 1.2M │ - │
│ Median Age │ 40.2 years │ 40.5 years │
│ Gender Ratio (M:F) │ 49:51 │ 49:51 │
│ Population Growth Rate │ +0.6% │ +0.5% │
│ Urban Population │ 68% │ 65% │
│ Rural Population │ 32% │ 35% │
└──────────────────────────────┴─────────────┴─────────────────┘

AGE DISTRIBUTION:
┌──────────────────┬─────────────┬──────────────┬──────────────┐
│ Age Group │ % of Pop │ Employment % │ Avg Income │
├──────────────────┼─────────────┼──────────────┼──────────────┤
│ 18-24 │ 10% │ 54% │ £18,500 │
│ 25-34 │ 17% │ 81% │ £32,500 │
│ 35-44 │ 16% │ 85% │ £42,800 │
│ 45-54 │ 15% │ 83% │ £45,600 │
│ 55-64 │ 15% │ 67% │ £37,200 │
│ 65+ │ 27% │ 12% │ £24,100 │
└──────────────────┴─────────────┴──────────────┴──────────────┘

REGIONAL BREAKDOWN:
┌─────────────┬──────────┬──────────┬──────────┬────────────┬─────────────┐
│ Region │ Med Age │ Unemp % │ Med Inc │ Homeowner %│ Health (1-5)│
├─────────────┼──────────┼──────────┼──────────┼────────────┼─────────────┤
│ City Centre │ 32.5 │ 8.2% │ £32,500 │ 28% │ 4.0 │
│ Suburbs │ 43.2 │ 3.8% │ £52,800 │ 72% │ 4.4 │
│ Market Towns│ 41.5 │ 4.5% │ £38,200 │ 68% │ 4.1 │
│ Villages │ 52.4 │ 3.2% │ £41,500 │ 79% │ 3.8 │
│ Coastal │ 54.8 │ 6.1% │ £29,700 │ 71% │ 3.5 │
└─────────────┴──────────┴──────────┴──────────┴────────────┴─────────────┘

WORKFORCE METRICS:
┌──────────────────────────┬─────────────┬─────────────────┐
│ Metric │ Value │ Demographic Note│
├──────────────────────────┼─────────────┼─────────────────┤
│ Overall Unemployment │ 5.2% │ Highest: 18-24 │
│ Underemployment Rate │ 9.1% │ (part-time wanting full-time)│
│ Remote Work % │ 28% │ Peak: 35-44 │
│ Gig Economy % │ 11% │ Peak: 18-34 │
│ Self-Employed % │ 14% │ Peak: 45-54 │
│ Public Sector % │ 22% │ Peak: Coastal │
│ Job Satisfaction (1-5) │ 3.7 │ Lowest: 18-24 │
└──────────────────────────┴─────────────┴─────────────────┘

EDUCATION & SKILLS:
┌──────────────────┬─────────────┬──────────────┬──────────────┐
│ Qualification │ % of Pop │ Avg Income │ Unemp % │
├──────────────────┼─────────────┼──────────────┼──────────────┤
│ No Qualifications│ 7% │ £17,200 │ 11.8% │
│ GCSEs (Level 2) │ 24% │ £23,500 │ 6.2% │
│ A-Levels (Level 3)│ 28% │ £28,400 │ 4.8% │
│ Bachelor's │ 26% │ £42,100 │ 2.9% │
│ Postgraduate │ 15% │ £54,600 │ 1.7% │
└──────────────────┴─────────────┴──────────────┴──────────────┘

HEALTH & WELLBEING:
┌──────────────────────────┬─────────────┬─────────────────┐
│ Metric │ Value │ At-Risk Groups │
├──────────────────────────┼─────────────┼─────────────────┤
│ Self-Reported Health (1-5│ 3.9 │ Coastal: 3.5 │
│ NHS Access Satisfaction %│ 68% │ Rural: 54% │
│ Mental Health Concerns % │ 21% │ 18-34: 28% │
│ Long-term Condition % │ 34% │ 65+: 61% │
│ Private Health Ins % │ 11% │ Suburbs: 18% │
│ Life Expectancy │ 80.1 years │ Coastal: 77.8 │
└──────────────────────────┴─────────────┴─────────────────┘

INCOME DISTRIBUTION:
┌──────────────────┬─────────────┬──────────────┬─────────────────┐
│ Income Quintile │ Income Range│ Avg Age │ Primary Regions │
├──────────────────┼─────────────┼──────────────┼─────────────────┤
│ Bottom 20% │ <£15,000 │ 44 │ Coastal/City │
│ Second 20% │ £15-24,000 │ 39 │ Mixed │
│ Middle 20% │ £24-35,000 │ 42 │ Market Towns │
│ Fourth 20% │ £35-52,000 │ 45 │ Suburbs/Villages│
│ Top 20% │ >£52,000 │ 48 │ Suburbs │
└──────────────────┴─────────────┴──────────────┴─────────────────┘

HOUSING METRICS:
┌──────────────────────────┬─────────────┬─────────────────┐
│ Metric │ Value │ Regional Note │
├──────────────────────────┼─────────────┼─────────────────┤
│ Average House Price │ £285,000 │ City: £320,000 │
│ Avg Monthly Rent (2-bed) │ £950 │ City: £1,250 │
│ Housing Benefit Claims % │ 14% │ Coastal: 22% │
│ Social Housing Wait List │ 8,400 │ Up 24% in 3yrs │
│ First-time Buyer Age │ 33 │ Up from 29 (2019)│
│ Council Tax Band D Avg │ £2,150 │ │
└──────────────────────────┴─────────────┴─────────────────┘

CORRELATIONS (Pearson coefficients):
┌─────────────────────────────────┬─────────┐
│ Variables │ r-value │
├─────────────────────────────────┼─────────┤
│ Education vs. Income │ +0.68 │
│ Income vs. Health │ +0.52 │
│ Age vs. Health │ -0.51 │
│ Coastal vs. Mental Health │ -0.28 │
│ Qualifications vs. Unemployment │ -0.61 │
│ House Prices vs. Age │ +0.58 │
│ Remote Work vs. Income │ +0.44 │
│ Rural vs. NHS Access │ -0.62 │
└─────────────────────────────────┴─────────┘



Based on this UK demographic data, please provide:

    KEY DEMOGRAPHIC INSIGHTS (identify 5 most important findings)

        What are the major trends and patterns?

        Which groups are thriving vs. struggling?

        What's changing over time?


Be specific, reference the statistics directly, and think like a UK policy analyst considering both economic and social factors. Use £ not $.
<|end|>
<|assistant|>"""

In [ ]:
llm = Llama(
    model_path="Mistral-7B-Instruct-v0.3-Q4_K_M.gguf",
    n_ctx=32768,              # Mistral's full 32K context window!
    n_threads=None,            # Auto-detect all CPU cores
    n_gpu_layers=0,            # CPU only (0 = no GPU offloading)
    n_batch=512,               # Batch size for processing
    verbose=False              # Set to True if you want to see detailed info
)

print("\n" + "="*60)
print("📊 Running Statistical Data Analysis Test...")
print("="*60 + "\n")

start = time.time()
response = llm(full_prompt, max_tokens=2500)  # Allow for detailed response
analysis = response["choices"][0]["text"].strip()

print(analysis)
print(f"\n{'='*60}")
print(f"Analysis completed in {time.time()-start:.1f} seconds")